# Create OCR Eval Dataset

Builds a [Weave Dataset](https://weave-docs.wandb.ai/guides/core-types/datasets) for OCR evaluation from the local `images/` and `text/` directories.

- Each row: `{id, image, ground_truth_text}`
- Images and ground-truth markdown files are paired by filename stem (`page_1.jpeg` ↔ `page_1.md`)
- Published to the W&B project defined in `.env` (`ENTITY`/`PROJECT`)

In [7]:
import logging
import os
from pathlib import Path

import pandas as pd
import weave
from dotenv import load_dotenv
from PIL import Image

logging.getLogger("weave").setLevel(logging.ERROR)

load_dotenv(override=True)
ENTITY = os.environ["ENTITY"]
PROJECT = os.environ["PROJECT"]
print(f"Target project: {ENTITY}/{PROJECT}")

Target project: wandb-smle/kateb


## Pair images with ground-truth text

Match every image in `images/` to its transcription in `text/` by filename stem, and fail loudly if anything is missing or unmatched.

In [3]:
IMAGES_DIR = Path("images")
TEXT_DIR = Path("text")

image_paths = {p.stem: p for p in sorted(IMAGES_DIR.glob("*.jpeg"))}
text_paths = {p.stem: p for p in sorted(TEXT_DIR.glob("*.md"))}

missing_text = set(image_paths) - set(text_paths)
missing_images = set(text_paths) - set(image_paths)
assert not missing_text, f"Images without ground truth: {missing_text}"
assert not missing_images, f"Ground truth without images: {missing_images}"

pairs = [(stem, image_paths[stem], text_paths[stem]) for stem in sorted(image_paths)]
print(f"Matched {len(pairs)} image/text pairs")

Matched 7 image/text pairs


## Preview

Sanity-check the pairing before publishing.

In [4]:
preview_df = pd.DataFrame(
    [
        {
            "id": stem,
            "image_file": img_path.name,
            "image_size": Image.open(img_path).size,
            "text_file": txt_path.name,
            "text_chars": len(txt_path.read_text()),
            "text_snippet": txt_path.read_text()[:100].replace("\n", " "),
        }
        for stem, img_path, txt_path in pairs
    ]
)
preview_df

,id,image_file,image_size,text_file,text_chars,text_snippet
0,IMG_2555,IMG_2555.jpeg,"(5712, 4284)",IMG_2555.md,903,# Rejection Sampling Involves curating new mo...
1,IMG_2556,IMG_2556.jpeg,"(5712, 4284)",IMG_2556.md,808,# Generating Completions To generate a set of...
2,IMG_2557,IMG_2557.jpeg,"(5712, 4284)",IMG_2557.md,1319,There are many ways to use $R$ to select top c...
3,IMG_2558,IMG_2558.jpeg,"(5712, 4284)",IMG_2558.md,343,"With the selected completions, we can perform ..."
4,page_1,page_1.jpeg,"(1200, 1600)",page_1.md,1294,## Morphing Probability distributions - Let $...
5,page_2,page_2.jpeg,"(1200, 1600)",page_2.md,897,- A probability path is a continuous sequence ...
6,page_3,page_3.jpeg,"(1200, 1600)",page_3.md,1553,# Training Loop - Given a minibatch of real d...


## Build dataset rows

Weave serializes PIL images natively, so each row carries the image object directly alongside the ground-truth text.

In [5]:
rows = [
    {
        "id": stem,
        "image": Image.open(img_path),
        "ground_truth_text": txt_path.read_text(),
    }
    for stem, img_path, txt_path in pairs
]
print(f"Built {len(rows)} rows with fields: {list(rows[0].keys())}")

Built 7 rows with fields: ['id', 'image', 'ground_truth_text']


## Publish to Weave

In [8]:
client = weave.init(f"{ENTITY}/{PROJECT}")

dataset = weave.Dataset(name="ocr-eval", rows=rows)
ref = weave.publish(dataset)

print(f"Published: {ref.uri()}")
print(f"View in UI: https://wandb.ai/{ENTITY}/{PROJECT}/weave/datasets")

weave: weave version 0.53.11 is available!  To upgrade, please run:
weave:  $ pip install weave --upgrade
weave: Logged in as Weights & Biases user: jbutt.
weave: View Weave data at https://wandb.ai/wandb-smle/kateb/weave
weave: 📦 Published to https://wandb.ai/wandb-smle/kateb/weave/objects/ocr-eval/versions/xNdEURcGBR8ZOgMDSxzm3678koUr6s8VTFJxfWlsZmI


Published: weave:///wandb-smle/kateb/object/ocr-eval:xNdEURcGBR8ZOgMDSxzm3678koUr6s8VTFJxfWlsZmI
View in UI: https://wandb.ai/wandb-smle/kateb/weave/datasets


## Verify round-trip

Fetch the published dataset back and confirm row count and content integrity.

In [6]:
fetched = weave.ref(f"weave:///{ENTITY}/{PROJECT}/object/ocr-eval:latest").get()
fetched_rows = list(fetched.rows)

assert len(fetched_rows) == len(pairs), f"Expected {len(pairs)} rows, got {len(fetched_rows)}"

first = fetched_rows[0]
expected_text = pairs[0][2].read_text()
assert first["ground_truth_text"] == expected_text, "Ground truth text mismatch"
assert first["image"].size == Image.open(pairs[0][1]).size, "Image size mismatch"

print(f"Verified {len(fetched_rows)} rows — first row id={first['id']}, "
      f"image={first['image'].size}, text={len(first['ground_truth_text'])} chars")

Verified 7 rows — first row id=IMG_2555, image=(5712, 4284), text=903 chars


# Save Dataset as a DataFrame

In [ ]:
try:
    [i.pop('id') for i in rows]
except KeyError:
    pass

ocr_df = pd.DataFrame(rows)



In [ ]:
Path.mkdir(Path("./dataset/"), parents=True, exist_ok=True)

ocr_df.to_csv("./dataset/ocr_eval_dataset.csv", index=False)